<a href="https://colab.research.google.com/github/Madhavi-1234/Machine-Learning-Practice/blob/main/fake_news_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import zipfile
import os

zip_path = "/content/WELFake_Dataset.csv.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

print(os.listdir("/content"))

['.config', 'WELFake_Dataset.csv', 'WELFake_Dataset.csv.zip', 'sample_data']


In [31]:
import pandas as pd
df= pd.read_csv("/content/WELFake_Dataset.csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [11]:
df.isnull().sum()

,0
Unnamed: 0,0
title,558
text,39
label,0


In [22]:
missing_percent = df.isnull().mean() * 100
missing_percent

,0
Unnamed: 0,0.000000
title,0.773560
text,0.054066
label,0.000000


In [23]:
df[df['title'].isna()][['title', 'text', 'label']].head()

,title,text,label
1,NaN,Did they post their votes for Hillary already?,1
43,NaN,True. Hillary needs a distraction and what bet...,1
162,NaN,All eyes on Electoral delegates. The People kn...,1
185,NaN,Cool,1
269,NaN,A leading US senator: US Supporting War in Syr...,1


In [24]:
df[df['text'].isna()][['title', 'text', 'label']].head()

,title,text,label
2457,Après le succès de « Mariés au premier regard ...,NaN,1
3534,Elections US : les premières estimations donne...,NaN,1
3709,110% des Américains assurent qu’ils continuero...,NaN,1
5612,Des millions d’Américains recherchent massivem...,NaN,1
6270,Vladimir Poutine est élu 45e Président des Eta...,NaN,1


title,False,True
text,,
False,0.99226,0.00774
True,1.00000,0.00000


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  72134 non-null  int64 
 1   title       71576 non-null  object
 2   text        72095 non-null  object
 3   label       72134 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 2.2+ MB


In [13]:
df.columns

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')

In [15]:
df.duplicated().sum()

np.int64(0)

In [18]:
# !pip install pandas-profiling


In [ ]:
# from pandas_profiling import ProfileReport
# profile = ProfileReport(df, title="Profiling Report")
# profile.to_file("report.html")

In [ ]:
# tackling  missing values  fro column text and label

In [20]:
df[df['text'].isna()][['title', 'text', 'label']].sum()


,0
title,Après le succès de « Mariés au premier regard ...
text,0
label,39


In [26]:
pd.crosstab(
    df['text'].isna(),
    df['title'].isna(),
    normalize='index'
)

title,False,True
text,,
False,0.99226,0.00774
True,1.00000,0.00000


In [45]:
print(df.shape)

(63678, 4)


In [36]:
print(df.columns)

Index(['title', 'text', 'label'], dtype='object')


In [37]:
# Remove duplicate records
df.drop_duplicates(inplace=True)


In [38]:
# Label is target → cannot be imputed
df.dropna(subset=['label'], inplace=True)

In [39]:
# Preserve rows even when title/text is missing
df['title']= df['title'].fillna('')
df['text']= df['text'].fillna('')

In [40]:
# Combine both sources of textual information
df['combined_text'] = df['title'] + ' ' + df['text']

In [41]:
df.isnull().sum()

,0
title,0
text,0
label,0
combined_text,0


In [42]:
df[df['text'].notna()]['title'].isna().value_counts()

,count
title,
False,63678


In [43]:
df[df['title'].notna()]['text'].isna().value_counts()

,count
text,
False,63678


In [44]:
(df['combined_text'].str.strip() == '').sum()

np.int64(0)

In [46]:
df[['title', 'text', 'combined_text', 'label']].head()

,title,text,combined_text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,,Did they post their votes for Hillary already?,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",SATAN 2: Russia unvelis an image of its terrif...,1


# **EDA**

In [47]:
df['label'].value_counts()

,count
label,
0,34791
1,28887


In [48]:
df['label'].value_counts(normalize= True)* 100

,proportion
label,
0,54.635824
1,45.364176


In [50]:
## check the length

df['text_length']= df['text'].str.len()
df['text_length']

,text_length
0,5049
1,46
2,216
3,8010
4,1916
...,...
72127,1237
72129,4788
72130,3634
72131,2864


In [51]:
# distribution

df['text_length'].describe()

,text_length
count,63678.000000
mean,3276.752866
std,3636.269585
min,0.000000
25%,1428.000000
50%,2456.000000
75%,4095.000000
max,142961.000000


In [52]:
# mini checkup

df['text_length'].isna().sum()

np.int64(0)

In [53]:
df.loc[df['text_length'].idxmax(), ['title', 'text', 'text_length']]

,6445
title,Заседание Международного дискуссионного клуба ...
text,Заседание Международного дискуссионного клуба ...
text_length,142961


In [54]:
df[df['text_length'] == 0][['title', 'text', 'label']].head()

,title,text,label
2457,Après le succès de « Mariés au premier regard ...,,1
3534,Elections US : les premières estimations donne...,,1
3709,110% des Américains assurent qu’ils continuero...,,1
5612,Des millions d’Américains recherchent massivem...,,1
6270,Vladimir Poutine est élu 45e Président des Eta...,,1


In [55]:
df.groupby('label')['text_length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,34791.0,3498.913685,3332.885898,1.0,1476.0,2644.0,4763.5,85948.0
1,28887.0,3009.186243,3954.535358,0.0,1383.0,2234.0,3347.0,142961.0
